In [ ]:
import os
import math
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def zeta_stats_from_grid(psi_grid, lam_grid, Delta_grid, p0, eps=1e-12, LD_floor=1e-8):
    psi = np.asarray(psi_grid, dtype=float)
    lam = np.asarray(lam_grid, dtype=float)
    Dlt = np.asarray(Delta_grid, dtype=float)
    p0 = float(p0)

    LD = lam * Dlt
    P = p0 * psi * LD
    dP = np.gradient(P, psi, edge_order=1)

    mask = np.isfinite(LD) & (LD >= LD_floor) & np.isfinite(dP)
    zeta = np.full_like(LD, np.nan, dtype=float)
    zeta[mask] = dP[mask] / LD[mask]

    if not np.any(mask):
        return {
            "zeta_min": np.nan,
            "zeta_med": np.nan,
            "zeta_max": np.nan,
            "zeta_share_lt1": np.nan,
            "underpriced_all_zeta": np.nan,
            "zeta_masked_share": 1.0,
            "zeta_valid_share": 0.0,
        }, zeta, P, dP

    z_min, z_med, z_max = np.nanmin(zeta), np.nanmedian(zeta), np.nanmax(zeta)
    share_lt1 = float(np.nanmean(zeta < 1.0))
    underpriced_all = bool(np.nanmax(zeta) < 1.0)
    masked_share = float(1.0 - np.mean(mask))
    valid_share = float(np.mean(mask))

    return {
        "zeta_min": float(z_min),
        "zeta_med": float(z_med),
        "zeta_max": float(z_max),
        "zeta_share_lt1": share_lt1,
        "underpriced_all_zeta": underpriced_all,
        "zeta_masked_share": masked_share,
        "zeta_valid_share": valid_share,
    }, zeta, P, dP

# =========================
# mc_block_1 – Micro + Drag (+ MP corridor)
# =========================
MC_NAME = "mc_block_1"
RUN_TS = datetime.now().strftime("%y%m%d_%H%M")
BASE_OUT_DIR = "/content/drive/MyDrive/vsc_saves"
OUT_DIR = os.path.join(BASE_OUT_DIR, f"{MC_NAME}_{RUN_TS}")
os.makedirs(OUT_DIR, exist_ok=True)

SEED = 20260212
rng = np.random.default_rng(SEED)

MC_CONFIG = {
    "N": 1000,
    "psi_grid_n": 200,
    "tau_int": 0.9,
    "Gamma_range": (0.01, 0.05),
    "rho_bar_range": (1.0, 5.0),
    "a_range": (0.5, 2.0),
    "c_lambda_range": (0.1, 1.0),
    "eta_lambda_range": (1.01, 2.0),
    "c_Delta_range": (0.1, 1.0),
    "eta_Delta_range": (1.0, 2.0),
    "c_delta_range": (0.05, 0.5),
    "eta_delta_range": (1.0, 2.0),
    "psi_L_range": (0.0, 0.3),
    "Wbar_range": (1.0, 50.0),
    # Outside-domain extensions
    "c_Omega_range": (0.0, 1.0),
    "eta_Omega_range": (0.5, 2.0),
    "p0_range": (0.0, 1.5),
    # MP closure parameter
    "kappa_c_range": (0.05, 2.0),
    "sigma_range": (0.01, 0.5),
    "chi_range": (1.0, 100.0),
    # Numerical tolerances / validity policy
    "eps_solv": 1e-12,
    "tol_mono": 1e-10,
    "tol_kkt": 1e-10,
    "tol_g": 1e-12,
    "tol_spill": 1e-10,
    "tau_spill_dom": 0.80,
    "tau_drag_dom": 0.80,
    "max_bad_share": 0.05,  # 5% tolerance for bad grid points
}

DERIV_RHO_MIN = 1e-8  # below this, derivative-based objects treated as invalid

def draw_uniform(rng, low, high):
    return rng.uniform(low, high)

def draw_primitives_base(rng, cfg):
    rho_bar    = draw_uniform(rng, *cfg["rho_bar_range"])
    a          = draw_uniform(rng, *cfg["a_range"])
    b          = a / (2.0 * rho_bar)
    r0         = 0.0
    c_lambda   = draw_uniform(rng, *cfg["c_lambda_range"])
    eta_lambda = draw_uniform(rng, *cfg["eta_lambda_range"])
    c_Delta    = draw_uniform(rng, *cfg["c_Delta_range"])
    eta_Delta  = draw_uniform(rng, *cfg["eta_Delta_range"])
    c_delta    = draw_uniform(rng, *cfg["c_delta_range"])
    eta_delta  = draw_uniform(rng, *cfg["eta_delta_range"])
    psi_L      = draw_uniform(rng, *cfg["psi_L_range"])
    Wbar       = draw_uniform(rng, *cfg["Wbar_range"])
    Gamma      = draw_uniform(rng, *cfg["Gamma_range"])
    kappa_c    = draw_uniform(rng, *cfg["kappa_c_range"])
    sigma      = draw_uniform(rng, *cfg["sigma_range"])
    chi        = draw_uniform(rng, *cfg["chi_range"])
    c_Omega    = draw_uniform(rng, *cfg["c_Omega_range"])
    eta_Omega  = draw_uniform(rng, *cfg["eta_Omega_range"])
    p0         = draw_uniform(rng, *cfg["p0_range"])

    return {
        "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
        "c_lambda": c_lambda, "eta_lambda": eta_lambda,
        "c_Delta": c_Delta, "eta_Delta": eta_Delta,
        "c_delta": c_delta, "eta_delta": eta_delta,
        "psi_L": psi_L, "Wbar": Wbar, "Gamma": Gamma,
        "kappa_c": kappa_c,
        "sigma": sigma, "chi": chi,
        "c_Omega": c_Omega, "eta_Omega": eta_Omega,
        "p0": p0,
    }

def r_func(rho, r0, a, b):
    return r0 + a * rho - b * rho**2

def r_prime(rho, a, b):
    return a - 2.0 * b * rho

def r_second(b):
    return -2.0 * b

def lam_func(rho, c_lam, eta_lam):
    return 1.0 - np.exp(-c_lam * rho**eta_lam)

def lam_prime(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    invalid = (~np.isfinite(arr)) | (arr < DERIV_RHO_MIN)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = ~invalid
    if np.any(valid):
        f = np.exp(-c_lam * arr[valid]**eta_lam)
        g = c_lam * eta_lam * arr[valid]**(eta_lam - 1.0)
        out[valid] = f * g
    return out if arr.ndim > 0 else float(out)

def lam_second(rho, c_lam, eta_lam):
    arr = np.asarray(rho, dtype=float)
    invalid = (~np.isfinite(arr)) | (arr < DERIV_RHO_MIN)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = ~invalid
    if np.any(valid):
        f = np.exp(-c_lam * arr[valid]**eta_lam)
        g = c_lam * eta_lam * arr[valid]**(eta_lam - 1.0)
        g_prime = c_lam * eta_lam * (eta_lam - 1.0) * arr[valid]**(eta_lam - 2.0)
        out[valid] = f * (g_prime - g**2)
    return out if arr.ndim > 0 else float(out)

def Delta_func(rho, c_Delta, eta_Delta):
    return c_Delta * rho**eta_Delta

def Delta_prime(rho, c_Delta, eta_Delta):
    arr = np.asarray(rho, dtype=float)
    invalid = (~np.isfinite(arr)) | (arr < DERIV_RHO_MIN)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = ~invalid
    if np.any(valid):
        out[valid] = c_Delta * eta_Delta * arr[valid]**(eta_Delta - 1.0)
    return out if arr.ndim > 0 else float(out)

def Delta_second(rho, c_Delta, eta_Delta):
    arr = np.asarray(rho, dtype=float)
    invalid = (~np.isfinite(arr)) | (arr < DERIV_RHO_MIN)
    out = np.full_like(arr, np.nan, dtype=float)
    valid = ~invalid
    if np.any(valid):
        out[valid] = c_Delta * eta_Delta * (eta_Delta - 1.0) * arr[valid]**(eta_Delta - 2.0)
    return out if arr.ndim > 0 else float(out)

def delta_func(rho, c_delta, eta_delta):
    return c_delta * rho**eta_delta

def B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta):
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam = lam_func(rho, c_lam, eta_lam)
    D = Delta_func(rho, c_Delta, eta_Delta)
    D_p = Delta_prime(rho, c_Delta, eta_Delta)
    return lam_p * D + lam * D_p

def B_prime(rho, c_lam, eta_lam, c_Delta, eta_Delta):
    lam = lam_func(rho, c_lam, eta_lam)
    lam_p = lam_prime(rho, c_lam, eta_lam)
    lam_pp = lam_second(rho, c_lam, eta_lam)
    D = Delta_func(rho, c_Delta, eta_Delta)
    D_p = Delta_prime(rho, c_Delta, eta_Delta)
    D_pp = Delta_second(rho, c_Delta, eta_Delta)
    return lam_pp * D + 2.0 * lam_p * D_p + lam * D_pp

def solve_rho_star(psi, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta, tol=1e-8, max_iter=200, tol_kkt=1e-10):
    # FOC: r'(rho) = (1-psi) B(rho) with K normalized to 1
    def F(rho):
        return r_prime(rho, a, b) - (1.0 - psi) * B_func(rho, c_lam, eta_lam, c_Delta, eta_Delta)

    # KKT endpoint tests (explicit)
    B0 = B_func(0.0, c_lam, eta_lam, c_Delta, eta_Delta)
    F0 = r_prime(0.0, a, b) - (1.0 - psi) * B0 if np.isfinite(B0) else np.nan

    Bhi = B_func(rho_bar, c_lam, eta_lam, c_Delta, eta_Delta)
    Fhi = r_prime(rho_bar, a, b) - (1.0 - psi) * Bhi if np.isfinite(Bhi) else np.nan

    rho_lo_int = DERIV_RHO_MIN
    rho_hi = rho_bar

    f_lo_int = F(rho_lo_int)
    f_hi = F(rho_hi)

    # If high endpoint is nonfinite, solver is invalid (no safe inference)
    if not np.isfinite(f_hi):
        return np.nan, "nonfinite"

    # Interior bracket only if interior left endpoint is finite
    if np.isfinite(f_lo_int) and (f_lo_int * f_hi < 0.0):
        lo, hi = rho_lo_int, rho_hi
        for _ in range(max_iter):
            mid = 0.5 * (lo + hi)
            f_mid = F(mid)
            if not np.isfinite(f_mid):
                return np.nan, "nonfinite"
            if abs(f_mid) < tol or (hi - lo) < tol:
                return mid, "interior"
            if np.sign(f_mid) == np.sign(f_lo_int):
                lo, f_lo_int = mid, f_mid
            else:
                hi = mid
        return np.nan, "no_converge"

    # Same-sign cases
    # CASE 1: Marginal profit is positive everywhere (F > 0) — maximum at upper bound.
    if np.isfinite(f_lo_int) and (f_lo_int > 0.0) and (f_hi > 0.0):
        return rho_bar, "corner_high"

    # CASE 2: Marginal profit is negative everywhere (F < 0) — maximum at lower bound.
    if np.isfinite(f_lo_int) and (f_lo_int < 0.0) and (f_hi < 0.0):
        return 0.0, "corner_low"

    return np.nan, "no_bracket"

# ================================
# Block 1c MP tools (closure)
# ================================

def g_star_stationary(psi, Gamma_eff, kappa_c):
    """
    Stationary ceiling g* solves:
        kappa_c * psi * g^2 + g - Gamma_eff = 0,
    where Gamma_eff = Gamma - D(psi).
    Positive root. If psi <= 0, g* = Gamma_eff.
    """
    if psi <= 0.0:
        return Gamma_eff
    disc = 1.0 + 4.0 * kappa_c * psi * Gamma_eff
    if disc < 0.0:
        return np.nan
    return (-1.0 + np.sqrt(disc)) / (2.0 * kappa_c * psi)

def wedge_threshold_stats_from_grids(psi_grid, rK_grid, g_star_grid, mask=None, zero_tol=1e-10):
    """
    Computes crossing diagnostics for W(psi) = rK(psi) - g(psi).

    If mask is provided (boolean array), diagnostics are computed on the masked subset
    in the original order.

    Returns: (stats_dict, W_array_used_for_stats)
    """
    psi = np.asarray(psi_grid, dtype=float)
    rK = np.asarray(rK_grid, dtype=float)
    gstar = np.asarray(g_star_grid, dtype=float)

    if mask is not None:
        mask = np.asarray(mask, dtype=bool)
        psi = psi[mask]
        rK = rK[mask]
        gstar = gstar[mask]

    if psi.size < 2:
        stats = {
            "endpoints_straddle": False,
            "unique_crossing": False,
            "psi_bar_hat": np.nan,
            "W_min": np.nan,
            "W_max": np.nan,
            "W_at_start": np.nan,
            "W_at_end": np.nan,
            "n_crossings": 0,
        }
        return stats, np.array([], dtype=float)

    W = rK - gstar

    Wmin = float(np.nanmin(W))
    Wmax = float(np.nanmax(W))
    W0 = float(W[0])
    W1 = float(W[-1])

    S = np.sign(np.where(np.abs(W) <= zero_tol, 0.0, W))
    endpoints_straddle = (S[0] == 0) or (S[-1] == 0) or (S[0] * S[-1] < 0)

    sign_changes = np.where(S[:-1] * S[1:] < 0)[0]
    unique_crossing = (len(sign_changes) == 1)

    psi_bar_hat = np.nan
    if unique_crossing:
        i = int(sign_changes[0])
        psi_lo, psi_hi = psi[i], psi[i + 1]
        W_lo, W_hi = W[i], W[i + 1]
        if abs(W_hi - W_lo) > 0:
            psi_bar_hat = float(psi_lo - W_lo * (psi_hi - psi_lo) / (W_hi - W_lo))

    stats = {
        "endpoints_straddle": bool(endpoints_straddle),
        "unique_crossing": bool(unique_crossing),
        "psi_bar_hat": psi_bar_hat,
        "W_min": Wmin,
        "W_max": Wmax,
        "W_at_start": W0,
        "W_at_end": W1,
        "n_crossings": int(len(sign_changes)),
    }
    return stats, W

def run_block1c_draw(rng_local, cfg, theta=None):
    """
    One MC draw for Block 1c:
    - Draw primitives + kappa_c.
    - Solve micro FOC on psi grid.
    - Construct MP corridor where BOTH
        (i) solvency holds and
        (ii) g*(psi, Gamma_eff) >= 0 is well-defined.
    - Impose interior/SOC/KKT/sign restrictions on that corridor.
    - Return primitives + MP corridor diagnostics.
    """
    if theta is None:
        theta = draw_primitives_base(rng_local, cfg)

    rho_bar    = theta["rho_bar"]
    a          = theta["a"]
    b          = theta["b"]
    r0         = theta["r0"]
    c_lambda   = theta["c_lambda"]
    eta_lambda = theta["eta_lambda"]
    c_Delta    = theta["c_Delta"]
    eta_Delta  = theta["eta_Delta"]
    c_delta    = theta["c_delta"]
    eta_delta  = theta["eta_delta"]
    psi_L      = theta["psi_L"]
    Wbar       = theta["Wbar"]
    Gamma      = theta["Gamma"]
    kappa_c    = theta["kappa_c"]
    sigma      = theta["sigma"]
    chi        = theta["chi"]

    tol_kkt = cfg.get("tol_kkt", 1e-10)
    tol_mono = cfg.get("tol_mono", 1e-10)
    eps_solv = cfg.get("eps_solv", 1e-12)
    tol_g = cfg.get("tol_g", 1e-12)
    max_bad_share = cfg.get("max_bad_share", 0.05)

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])

    # Baseline drag at psi=0 (uses delta, not Delta)
    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lambda, eta_lambda, c_Delta, eta_Delta, tol_kkt=tol_kkt)
    if st0 not in {"interior", "corner_low", "corner_high"}:
        return {
            "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
            "c_lambda": c_lambda, "eta_lambda": eta_lambda,
            "c_Delta": c_Delta, "eta_Delta": eta_Delta,
            "c_delta": c_delta, "eta_delta": eta_delta,
            "psi_L": psi_L, "Wbar": Wbar, "Gamma": Gamma,
            "kappa_c": kappa_c,
            "sigma": sigma, "chi": chi,
            "baseline_status": st0,
            "excluded_solver": True,
            "included": False,
            "solver_ok_share": 0.0,
            "first_invalid_idx": 0,
            "count_nonfinite": 0,
            "count_no_bracket": 0,
            "count_no_converge": 0,
            "count_corner_low": 0,
            "count_corner_high": 0,
            "count_interior": 0,
            "mp_domain_core": False,
            "wedge_regime": "failed_solver",
            "mp_exists": False,
            "mp_corr_share": 0.0,
            "corridor_has_corners": False,
            "corridor_corner_share": np.nan,
            "deriv_invalid_any": False,
            "deriv_invalid_count": 0,
            "deriv_invalid_idx_first": np.nan,
        }
    lam0    = lam_func(rho0, c_lambda, eta_lambda)
    del0    = delta_func(rho0, c_delta, eta_delta)
    base_drag = lam0 * del0

    rho_star = np.empty_like(psi_grid)
    solver_status = np.empty(psi_grid.size, dtype=object)
    solver_ok_grid = np.zeros(psi_grid.size, dtype=bool)
    is_interior = np.zeros(psi_grid.size, dtype=bool)
    is_corner_low = np.zeros(psi_grid.size, dtype=bool)
    is_corner_high = np.zeros(psi_grid.size, dtype=bool)
    soc_ok_grid = np.full(psi_grid.size, np.nan)
    kkt_ok_grid = np.full(psi_grid.size, np.nan)

    deriv_invalid_count = 0
    deriv_invalid_any = False
    deriv_invalid_idx = []

    rho_lo = 0.0
    for idx, psi in enumerate(psi_grid):
        rho_i, status_i = solve_rho_star(
            psi, rho_bar, a, b, c_lambda, eta_lambda, c_Delta, eta_Delta, tol_kkt=tol_kkt
        )
        solver_status[idx] = status_i
        solver_ok_grid[idx] = status_i in {"interior", "corner_low", "corner_high"}
        rho_star[idx] = rho_i if solver_ok_grid[idx] else np.nan

        is_interior[idx] = status_i == "interior"
        is_corner_low[idx] = status_i == "corner_low"
        is_corner_high[idx] = status_i == "corner_high"

        if status_i == "interior":
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                soc_ok_grid[idx] = np.nan
                deriv_invalid_count += 1
                deriv_invalid_any = True
                deriv_invalid_idx.append(idx)
            else:
                Bp = B_prime(rho_i, c_lambda, eta_lambda, c_Delta, eta_Delta)
                if not np.isfinite(Bp):
                    soc_ok_grid[idx] = np.nan
                    deriv_invalid_count += 1
                    deriv_invalid_any = True
                    deriv_invalid_idx.append(idx)
                else:
                    soc_val = r_second(b) - (1.0 - psi) * Bp
                    soc_ok_grid[idx] = np.isfinite(soc_val) and (soc_val < 0.0)
        elif status_i == "corner_high":
            F_hi = r_prime(rho_bar, a, b) - (1.0 - psi) * B_func(rho_bar, c_lambda, eta_lambda, c_Delta, eta_Delta)
            kkt_ok_grid[idx] = np.isfinite(F_hi) and (F_hi >= -tol_kkt)
        elif status_i == "corner_low":
            B_lo = B_func(rho_lo, c_lambda, eta_lambda, c_Delta, eta_Delta)
            if not np.isfinite(B_lo):
                kkt_ok_grid[idx] = False
            else:
                F_lo = r_prime(rho_lo, a, b) - (1.0 - psi) * B_lo
                kkt_ok_grid[idx] = np.isfinite(F_lo) and (F_lo <= +tol_kkt)
        else:
            kkt_ok_grid[idx] = False

    bad_share = 1.0 - solver_ok_grid.mean()
    excluded_solver = bad_share > max_bad_share
    included = not excluded_solver
    first_invalid_idx = np.nan
    if excluded_solver:
        bad_idx = np.where(~solver_ok_grid)[0]
        first_invalid_idx = int(bad_idx[0]) if bad_idx.size else np.nan

    count_nonfinite = int(np.sum(solver_status == "nonfinite"))
    count_no_bracket = int(np.sum(solver_status == "no_bracket"))
    count_no_converge = int(np.sum(solver_status == "no_converge"))
    count_corner_low = int(np.sum(solver_status == "corner_low"))
    count_corner_high = int(np.sum(solver_status == "corner_high"))
    count_interior = int(np.sum(solver_status == "interior"))

    lam_grid   = lam_func(rho_star, c_lambda, eta_lambda)
    Delta_grid = Delta_func(rho_star, c_Delta, eta_Delta)
    delta_grid = delta_func(rho_star, c_delta, eta_delta)

    drag_grid = lam_grid * delta_grid - base_drag
    rK_grid   = r_func(rho_star, r0, a, b) - (1.0 - psi_grid) * lam_grid * Delta_grid

    # MP closure: g*(psi, Gamma_eff)
    Gamma_eff_grid = Gamma - drag_grid
    g_star_grid = np.array([g_star_stationary(psi, Gamma_eff_grid[i], kappa_c) for i, psi in enumerate(psi_grid)])

    solv_grid = (Wbar - psi_grid * Delta_grid) > (eps_solv * max(1.0, Wbar))
    g_star_ok_grid = np.isfinite(g_star_grid) & (g_star_grid >= -tol_g)

    mp_feasible_grid = solver_ok_grid & solv_grid & g_star_ok_grid

    # MP corridor: maximal prefix starting at psi_L
    corridor_mask = np.zeros_like(psi_grid, dtype=bool)
    if mp_feasible_grid[0]:
        first_false = np.where(~mp_feasible_grid)[0]
        if first_false.size == 0:
            last_true = psi_grid.size - 1
        else:
            last_true = first_false[0] - 1
        if last_true >= 1:
            corridor_mask[: last_true + 1] = True

    # Wedge threshold diagnostic: corridor-only stats
    wstats_corr, Wstar_corr = wedge_threshold_stats_from_grids(
        psi_grid, rK_grid, g_star_grid, mask=corridor_mask, zero_tol=1e-10
    )

    # Re-entry diagnostics
    first_fail_idx = np.nan
    if np.any(~mp_feasible_grid):
        first_fail_idx = int(np.where(~mp_feasible_grid)[0][0])
    if np.any(mp_feasible_grid):
        psi_last_true_overall = psi_grid[mp_feasible_grid].max()
    else:
        psi_last_true_overall = np.nan
    reentry_count = 0
    if np.isfinite(first_fail_idx):
        start = int(first_fail_idx)
        after = mp_feasible_grid[start:]
        if after.size > 1:
            reentry_count = int(np.sum((~after[:-1]) & (after[1:])))
    reentry_exists = reentry_count > 0

    mp_corr_len   = corridor_mask.sum()
    mp_exists     = mp_corr_len >= 1
    mp_corr_share = mp_corr_len / psi_grid.size

    # Micro filters restricted to MP corridor
    interior_corr = is_interior & corridor_mask
    corner_corr = (is_corner_low | is_corner_high) & corridor_mask
    if mp_exists and mp_corr_len > 0:
        f_int_corr = interior_corr.sum() / mp_corr_len
    else:
        f_int_corr = 0.0
    interior_ok_corr = (f_int_corr >= cfg["tau_int"]) if mp_exists else False
    if interior_corr.any():
        soc_slice = soc_ok_grid[interior_corr]
        soc_ok_corr = bool(np.all(soc_slice == True)) and bool(np.all(np.isfinite(soc_slice)))
    else:
        soc_ok_corr = False
    kkt_ok_corr = np.all(kkt_ok_grid[corner_corr] == True) if corner_corr.any() else True
    micro_opt_ok_corr = soc_ok_corr and kkt_ok_corr

    dpsi   = psi_grid[1] - psi_grid[0] if psi_grid.size > 1 else np.nan
    d_rho  = np.gradient(rho_star, dpsi)
    d_drag = np.gradient(drag_grid, dpsi)
    d_rK   = np.gradient(rK_grid, dpsi)

    W_grid = rK_grid - g_star_grid
    dW = np.gradient(W_grid, dpsi)

    if mp_corr_len > 1:
        d_W_pos_corr = np.all(dW[corridor_mask] >= -tol_mono)
        mono_violation_share_W = float(np.mean(dW[corridor_mask] < -tol_mono))
    else:
        d_W_pos_corr = True
        mono_violation_share_W = np.nan

    if interior_corr.any() and mp_corr_len > 1:
        d_rho_pos_corr = np.all(d_rho[interior_corr]  >= -tol_mono)
        d_D_pos_corr   = np.all(d_drag[interior_corr] >= -tol_mono)
        d_rK_pos_corr  = np.all(d_rK[interior_corr]   >= -tol_mono)
        mono_violation_share_rho = float(np.mean(d_rho[interior_corr] < -tol_mono))
        mono_violation_share_D = float(np.mean(d_drag[interior_corr] < -tol_mono))
        mono_violation_share_rK = float(np.mean(d_rK[interior_corr] < -tol_mono))
    elif interior_corr.any():
        d_rho_pos_corr = True
        d_D_pos_corr   = True
        d_rK_pos_corr  = True
        mono_violation_share_rho = np.nan
        mono_violation_share_D = np.nan
        mono_violation_share_rK = np.nan
    else:
        d_rho_pos_corr = False
        d_D_pos_corr   = False
        d_rK_pos_corr  = False
        mono_violation_share_rho = np.nan
        mono_violation_share_D = np.nan
        mono_violation_share_rK = np.nan

    corridor_has_corners = bool(corner_corr.any())
    corridor_corner_share = float(corner_corr.sum() / mp_corr_len) if mp_corr_len > 0 else np.nan

    mp_domain_core = (
        included
        and (not deriv_invalid_any)
        and mp_exists
        and interior_ok_corr
        and micro_opt_ok_corr
        and d_rho_pos_corr
        and d_D_pos_corr
        and d_rK_pos_corr
        and d_W_pos_corr
    )

    if not mp_exists:
        wedge_regime = "no_corridor"
        mean_rho_corr     = np.nan
        max_drag_corr     = np.nan
        min_solvmargin    = np.nan
        g_star_min_corr   = np.nan
        g_star_max_corr   = np.nan
        slope_rK_mid_corr = np.nan
        psi_mp_max        = np.nan
    else:
        psi_mp_max        = psi_grid[corridor_mask].max()
        mean_rho_corr     = rho_star[corridor_mask].mean()
        max_drag_corr     = drag_grid[corridor_mask].max()
        min_solvmargin    = (Wbar - psi_grid * Delta_grid)[corridor_mask].min()
        g_star_min_corr   = g_star_grid[corridor_mask].min()
        g_star_max_corr   = g_star_grid[corridor_mask].max()
        slope_rK_mid_corr = np.interp(
            0.5 * (psi_L + 1.0), psi_grid, np.gradient(rK_grid, dpsi)
        )

        if wstats_corr["unique_crossing"]:
            wedge_regime = "crossing"
        elif wstats_corr["W_max"] < 0:
            wedge_regime = "always_stable"
        elif wstats_corr["W_min"] > 0:
            wedge_regime = "always_unstable"
        else:
            wedge_regime = "indeterminate"

    return {
        # primitives
        "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
        "c_lambda": c_lambda, "eta_lambda": eta_lambda,
        "c_Delta": c_Delta, "eta_Delta": eta_Delta,
        "c_delta": c_delta, "eta_delta": eta_delta,
        "psi_L": psi_L, "Wbar": Wbar, "Gamma": Gamma,
        "kappa_c": kappa_c,
        "sigma": sigma, "chi": chi,
        # solver diagnostics
        "baseline_status": st0,
        "solver_status": solver_status.tolist(),
        "solver_ok_share": 1.0 - bad_share,
        "excluded_solver": excluded_solver,
        "included": included,
        "first_invalid_idx": first_invalid_idx,
        "count_nonfinite": count_nonfinite,
        "count_no_bracket": count_no_bracket,
        "count_no_converge": count_no_converge,
        "count_corner_low": count_corner_low,
        "count_corner_high": count_corner_high,
        "count_interior": count_interior,
        "deriv_invalid_any": bool(deriv_invalid_any),
        "deriv_invalid_count": int(deriv_invalid_count),
        "deriv_invalid_idx_first": int(deriv_invalid_idx[0]) if deriv_invalid_idx else np.nan,
        # MP corridor flags
        "mp_exists": mp_exists,
        "mp_corr_share": mp_corr_share,
        "psi_mp": psi_mp_max,
        "f_int_corr": f_int_corr,
        "interior_ok_corr": interior_ok_corr,
        "soc_ok_corr": soc_ok_corr,
        "kkt_ok_corr": kkt_ok_corr,
        "micro_opt_ok_corr": micro_opt_ok_corr,
        "d_rho_pos_corr": d_rho_pos_corr,
        "d_D_pos_corr": d_D_pos_corr,
        "d_rK_pos_corr": d_rK_pos_corr,
        "d_W_pos_corr": d_W_pos_corr,
        "mono_violation_share_rho": mono_violation_share_rho,
        "mono_violation_share_D": mono_violation_share_D,
        "mono_violation_share_rK": mono_violation_share_rK,
        "mono_violation_share_W": mono_violation_share_W,
        "corridor_has_corners": corridor_has_corners,
        "corridor_corner_share": corridor_corner_share,
        "mp_domain_core": mp_domain_core,
        "wedge_regime": wedge_regime,
        # wedge diagnostics (stored)
        "endpoints_straddle": wstats_corr["endpoints_straddle"],
        "unique_crossing": wstats_corr["unique_crossing"],
        "psi_bar_hat": wstats_corr["psi_bar_hat"],
        "Wstar_min": wstats_corr["W_min"],
        "Wstar_max": wstats_corr["W_max"],
        "Wstar_at_psiL": wstats_corr["W_at_start"],
        "Wstar_at_end": wstats_corr["W_at_end"],
        "n_crossings": wstats_corr["n_crossings"],
        # corridor metrics
        "mean_rho_corr": mean_rho_corr,
        "max_drag_corr": max_drag_corr,
        "slope_rK_mid_corr": slope_rK_mid_corr,
        "min_solvmargin_corr": min_solvmargin,
        "g_star_min_corr": g_star_min_corr,
        "g_star_max_corr": g_star_max_corr,
        "first_fail_idx": first_fail_idx,
        "reentry_count": reentry_count,
        "reentry_exists": reentry_exists,
        "psi_last_true_overall": psi_last_true_overall,
    }

def run_block1c_mc(rng_local, cfg, thetas=None):
    if thetas is None:
        rows = [run_block1c_draw(rng_local, cfg) for _ in range(cfg["N"])]
    else:
        rows = [run_block1c_draw(rng_local, cfg, theta=t) for t in thetas]
    return pd.DataFrame(rows)

# =========================
# Original Block 1 (baseline)
# =========================

def run_block1_draw(rng, cfg, theta=None, with_spillovers=False, with_premium=False):
    # ---- Primitives ----
    if theta is None:
        theta = draw_primitives_base(rng, cfg)

    rho_bar = theta["rho_bar"]
    a = theta["a"]
    b = theta["b"]
    r0 = theta["r0"]
    c_lam = theta["c_lambda"]
    eta_lam = theta["eta_lambda"]
    c_Delta = theta["c_Delta"]
    eta_Delta = theta["eta_Delta"]
    c_delta = theta["c_delta"]
    eta_delta = theta["eta_delta"]
    psi_L = theta["psi_L"]
    Wbar = theta["Wbar"]
    Gamma = theta["Gamma"]

    if with_spillovers:
        c_Omega = theta["c_Omega"]
        eta_Omega = theta["eta_Omega"]
    else:
        c_Omega = 0.0
        eta_Omega = 1.0

    if with_premium:
        p0 = theta["p0"]
    else:
        p0 = 0.0

    tol_kkt = cfg.get("tol_kkt", 1e-10)
    tol_mono = cfg.get("tol_mono", 1e-10)
    tol_spill = cfg.get("tol_spill", tol_mono)
    eps_solv = cfg.get("eps_solv", 1e-12)
    max_bad_share = cfg.get("max_bad_share", 0.05)

    underpricing_regime = (p0 < 1.0) if with_premium else True

    psi_grid = np.linspace(psi_L, 1.0, cfg["psi_grid_n"])
    rho_star = np.empty_like(psi_grid)
    solver_status = np.empty(psi_grid.size, dtype=object)
    solver_ok_grid = np.zeros(psi_grid.size, dtype=bool)
    interior = np.zeros(psi_grid.size, dtype=bool)
    is_corner_low = np.zeros(psi_grid.size, dtype=bool)
    is_corner_high = np.zeros(psi_grid.size, dtype=bool)
    soc_ok_grid = np.full(psi_grid.size, np.nan)
    kkt_ok_grid = np.full(psi_grid.size, np.nan)
    drag_dom_grid = np.ones(psi_grid.size, dtype=bool)
    spill_grid = np.zeros(psi_grid.size, dtype=bool)
    gap_finite_grid = np.zeros(psi_grid.size, dtype=bool)

    deriv_invalid_count = 0
    deriv_invalid_any = False
    deriv_invalid_idx = []

    # Growth drag baseline at psi=0 uses delta, not Delta
    rho0, st0 = solve_rho_star(0.0, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta, tol_kkt=tol_kkt)
    if st0 not in {"interior", "corner_low", "corner_high"}:
        return {
            "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
            "c_lambda": c_lam, "eta_lambda": eta_lam,
            "c_Delta": c_Delta, "eta_Delta": eta_Delta,
            "c_delta": c_delta, "eta_delta": eta_delta,
            "psi_L": psi_L, "Wbar": Wbar, "Gamma": Gamma,
            "c_Omega": c_Omega, "eta_Omega": eta_Omega, "p0": p0,
            "baseline_status": st0,
            "excluded_solver": True,
            "included": False,
            "solver_ok_share": 0.0,
            "first_invalid_idx": 0,
            "count_nonfinite": 0,
            "count_no_bracket": 0,
            "count_no_converge": 0,
            "count_corner_low": 0,
            "count_corner_high": 0,
            "count_interior": 0,
            "domain_core_block1": False,
            "domain_outside_block1": False,
            "drag_dominant": False if with_spillovers else True,
            "drag_share": 1.0 if not with_spillovers else 0.0,
            "spill_share": 0.0,
            "spill_share_low25": 0.0,
            "spill_share_high25": 0.0,
            "spill_share_low10": 0.0,
            "spill_share_high10": 0.0,
            "spill_at_mid": False,
            "psi_cutoff_spill": np.nan,
            "mixed_spill": False,
            "spillover_dominant": False,
            "spill_params_active": with_spillovers,
            "zeta_min": np.nan,
            "zeta_med": np.nan,
            "zeta_max": np.nan,
            "zeta_share_lt1": np.nan,
            "underpriced_all_zeta": np.nan,
            "zeta_masked_share": np.nan,
            "zeta_valid_share": np.nan,
            "zeta_drKnet_alignment": np.nan,
            "drKnet_min": np.nan,
            "drKnet_med": np.nan,
            "underpricing_regime": underpricing_regime,
            "f_int": np.nan, "share_corner": np.nan, "avg_rho": np.nan,
            "max_drag": np.nan, "slope_rK_mid": np.nan,
            "d_rho_pos": np.nan, "d_D_pos": np.nan, "d_rK_pos": np.nan,
            "mono_violation_share_rho": np.nan,
            "mono_violation_share_D": np.nan,
            "mono_violation_share_rK": np.nan,
            "interior_ok": False, "feasibility_ok": False, "soc_ok": False,
            "kkt_ok": False, "micro_opt_ok": False,
            "Wstruct_endpoints_straddle": False,
            "Wstruct_unique_crossing": False,
            "Wstruct_psi_bar_hat": np.nan,
            "Wstruct_min": np.nan,
            "Wstruct_max": np.nan,
            "Wstruct_at_psiL": np.nan,
            "Wstruct_at_1": np.nan,
            "Wstruct_n_crossings": np.nan,
            "deriv_invalid_any": False,
            "deriv_invalid_count": 0,
            "deriv_invalid_idx_first": np.nan,
        }
    lam0 = lam_func(rho0, c_lam, eta_lam)
    del0 = delta_func(rho0, c_delta, eta_delta)
    base_drag = lam0 * del0

    rho_lo = 0.0
    for idx, psi in enumerate(psi_grid):
        rho_i, status_i = solve_rho_star(
            psi, rho_bar, a, b, c_lam, eta_lam, c_Delta, eta_Delta, tol_kkt=tol_kkt
        )
        solver_status[idx] = status_i
        solver_ok_grid[idx] = status_i in {"interior", "corner_low", "corner_high"}
        rho_star[idx] = rho_i if solver_ok_grid[idx] else np.nan
        interior[idx] = status_i == "interior"
        is_corner_low[idx] = status_i == "corner_low"
        is_corner_high[idx] = status_i == "corner_high"

        if status_i == "interior":
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                soc_ok_grid[idx] = np.nan
                deriv_invalid_count += 1
                deriv_invalid_any = True
                deriv_invalid_idx.append(idx)
            else:
                Bp = B_prime(rho_i, c_lam, eta_lam, c_Delta, eta_Delta)
                if not np.isfinite(Bp):
                    soc_ok_grid[idx] = np.nan
                    deriv_invalid_count += 1
                    deriv_invalid_any = True
                    deriv_invalid_idx.append(idx)
                else:
                    soc_val = r_second(b) - (1.0 - psi) * Bp
                    soc_ok_grid[idx] = np.isfinite(soc_val) and (soc_val < 0.0)
        elif status_i == "corner_high":
            F_hi = r_prime(rho_bar, a, b) - (1.0 - psi) * B_func(rho_bar, c_lam, eta_lam, c_Delta, eta_Delta)
            kkt_ok_grid[idx] = np.isfinite(F_hi) and (F_hi >= -tol_kkt)
        elif status_i == "corner_low":
            B_lo = B_func(rho_lo, c_lam, eta_lam, c_Delta, eta_Delta)
            if not np.isfinite(B_lo):
                kkt_ok_grid[idx] = False
            else:
                F_lo = r_prime(rho_lo, a, b) - (1.0 - psi) * B_lo
                kkt_ok_grid[idx] = np.isfinite(F_lo) and (F_lo <= +tol_kkt)
        else:
            kkt_ok_grid[idx] = False

        if with_spillovers and solver_ok_grid[idx]:
            if (not np.isfinite(rho_i)) or (rho_i < DERIV_RHO_MIN):
                Omega_p = np.nan
                deriv_invalid_any = True
                deriv_invalid_count += 1
                deriv_invalid_idx.append(idx)
            else:
                Omega_p = c_Omega * eta_Omega * rho_i**(eta_Omega - 1.0)

            B_i = B_func(rho_i, c_lam, eta_lam, c_Delta, eta_Delta)
            gap = Omega_p - psi * B_i if (np.isfinite(Omega_p) and np.isfinite(B_i)) else np.nan

            gap_finite = np.isfinite(gap)
            gap_finite_grid[idx] = gap_finite
            # Only classify where gap is finite; otherwise leave both False
            if gap_finite:
                drag_dom_grid[idx] = (gap <= tol_spill)
                spill_grid[idx] = (gap > tol_spill)
            else:
                drag_dom_grid[idx] = False
                spill_grid[idx] = False
        elif with_spillovers:
            drag_dom_grid[idx] = False
            spill_grid[idx] = False

    bad_share = 1.0 - solver_ok_grid.mean()
    excluded_solver = bad_share > max_bad_share
    included = not excluded_solver
    first_invalid_idx = np.nan
    if excluded_solver:
        bad_idx = np.where(~solver_ok_grid)[0]
        first_invalid_idx = int(bad_idx[0]) if bad_idx.size else np.nan

    count_nonfinite = int(np.sum(solver_status == "nonfinite"))
    count_no_bracket = int(np.sum(solver_status == "no_bracket"))
    count_no_converge = int(np.sum(solver_status == "no_converge"))
    count_corner_low = int(np.sum(solver_status == "corner_low"))
    count_corner_high = int(np.sum(solver_status == "corner_high"))
    count_interior = int(np.sum(solver_status == "interior"))

    lam_grid = lam_func(rho_star, c_lam, eta_lam)
    Delta_grid = Delta_func(rho_star, c_Delta, eta_Delta)
    delta_grid = delta_func(rho_star, c_delta, eta_delta)
    drag_grid = lam_grid * delta_grid - base_drag
    rK_grid = r_func(rho_star, r0, a, b) - (1.0 - psi_grid) * lam_grid * Delta_grid
    g_grid = Gamma - drag_grid

    if with_premium:
        rK_net_grid = rK_grid - p0 * psi_grid * lam_grid * Delta_grid
    else:
        rK_net_grid = rK_grid

    wstruct_stats, _ = wedge_threshold_stats_from_grids(
        psi_grid, rK_net_grid, g_grid, mask=None, zero_tol=1e-10
    )

    dpsi = psi_grid[1] - psi_grid[0] if psi_grid.size > 1 else np.nan
    d_rho = np.gradient(rho_star, dpsi)
    d_drag = np.gradient(drag_grid, dpsi)
    d_rK = np.gradient(rK_net_grid, dpsi)

    if with_premium:
        zstats, zeta_grid, P_grid, dP_grid = zeta_stats_from_grid(psi_grid, lam_grid, Delta_grid, p0)
        tol_z = 1e-10

        valid = np.isfinite(zeta_grid) & np.isfinite(d_rK)
        if valid.any():
            good = (((zeta_grid < 1.0) & (d_rK > -tol_z)) |
                    ((zeta_grid >= 1.0) & (d_rK <  tol_z)))
            zeta_drKnet_alignment = float(np.mean(good[valid]))
            drKnet_min = float(np.nanmin(d_rK[valid]))
            drKnet_med = float(np.nanmedian(d_rK[valid]))
        else:
            zeta_drKnet_alignment = np.nan
            drKnet_min = np.nan
            drKnet_med = np.nan
    else:
        zstats = {
            "zeta_min": np.nan,
            "zeta_med": np.nan,
            "zeta_max": np.nan,
            "zeta_share_lt1": np.nan,
            "underpriced_all_zeta": np.nan,
            "zeta_masked_share": np.nan,
            "zeta_valid_share": np.nan,
        }
        zeta_drKnet_alignment = np.nan
        drKnet_min = np.nan
        drKnet_med = np.nan

    interior_mask = interior
    corner_mask = is_corner_low | is_corner_high
    if interior_mask.any():
        d_rho_pos = np.all(d_rho[interior_mask] >= -tol_mono)
        d_D_pos = np.all(d_drag[interior_mask] >= -tol_mono)
        d_rK_pos = np.all(d_rK[interior_mask] >= -tol_mono)
        mono_violation_share_rho = float(np.mean(d_rho[interior_mask] < -tol_mono))
        mono_violation_share_D = float(np.mean(d_drag[interior_mask] < -tol_mono))
        mono_violation_share_rK = float(np.mean(d_rK[interior_mask] < -tol_mono))
    else:
        d_rho_pos = False
        d_D_pos = False
        d_rK_pos = False
        mono_violation_share_rho = np.nan
        mono_violation_share_D = np.nan
        mono_violation_share_rK = np.nan

    f_int = interior_mask.mean()
    share_corner = 1.0 - f_int
    if interior_mask.any():
        soc_slice = soc_ok_grid[interior_mask]
        soc_ok = bool(np.all(soc_slice == True)) and bool(np.all(np.isfinite(soc_slice)))
    else:
        soc_ok = False
    kkt_ok = np.all(kkt_ok_grid[corner_mask] == True) if corner_mask.any() else True
    micro_opt_ok = soc_ok and kkt_ok
    feasibility_ok = np.all((Wbar - psi_grid * Delta_grid) > (eps_solv * max(1.0, Wbar)))

    if with_spillovers:
        valid_gap = solver_ok_grid & gap_finite_grid

        # Diagnostic shares (for reporting)
        drag_share = float(drag_dom_grid[valid_gap].mean()) if valid_gap.any() else np.nan
        spill_share = float(spill_grid[valid_gap].mean()) if valid_gap.any() else np.nan

        # PAPER DEFINITIONS (NO THRESHOLDS):
        # drag-dominant: inequality holds everywhere on the grid
        drag_dominant = bool(np.all(drag_dom_grid[valid_gap])) if valid_gap.any() else False

        # spillover-dominant: violated at least once on the grid
        spillover_dominant = bool(np.any(spill_grid[valid_gap])) if valid_gap.any() else False

        # stronger notion: spillovers win everywhere
        spillover_always = bool(np.all(spill_grid[valid_gap])) if valid_gap.any() else False

        if not valid_gap.any():
            spillover_regime = "indeterminate"
        elif drag_dominant:
            spillover_regime = "always_drag"
        elif spillover_always:
            spillover_regime = "always_spill"
        else:
            spillover_regime = "mixed"
        mixed_spill = spillover_regime == "mixed"

        if spillover_dominant:
            psi_cutoff_spill = float(psi_grid[np.where(spill_grid)[0].max()])
        else:
            psi_cutoff_spill = np.nan

        n = psi_grid.size
        k25 = max(1, int(np.floor(0.25 * n)))
        k10 = max(1, int(np.floor(0.10 * n)))
        spill_share_low25 = float(spill_grid[:k25].mean())
        spill_share_high25 = float(spill_grid[-k25:].mean())
        spill_share_low10 = float(spill_grid[:k10].mean())
        spill_share_high10 = float(spill_grid[-k10:].mean())
        spill_at_mid = bool(spill_grid[n // 2])
    else:
        drag_share = 1.0
        spill_share = 0.0
        spill_share_low25 = 0.0
        spill_share_high25 = 0.0
        spill_share_low10 = 0.0
        spill_share_high10 = 0.0
        spill_at_mid = False
        psi_cutoff_spill = np.nan
        mixed_spill = False
        drag_dominant = True
        spillover_dominant = False

    interior_ok = f_int >= cfg["tau_int"]
    domain_core_block1 = (
        included
        and (not deriv_invalid_any)
        and interior_ok
        and feasibility_ok
        and micro_opt_ok
        and underpricing_regime
    )
    domain_outside_block1 = included and (not domain_core_block1)

    mid_idx = len(psi_grid) // 2
    slope_rK_mid = d_rK[mid_idx] if len(psi_grid) > 2 else np.nan
    max_drag = float(np.max(drag_grid))
    avg_rho = float(np.mean(rho_star))

    return {
        "rho_bar": rho_bar, "a": a, "b": b, "r0": r0,
        "c_lambda": c_lam, "eta_lambda": eta_lam,
        "c_Delta": c_Delta, "eta_Delta": eta_Delta,
        "c_delta": c_delta, "eta_delta": eta_delta,
        "psi_L": psi_L, "Wbar": Wbar, "Gamma": Gamma,
        "c_Omega": c_Omega, "eta_Omega": eta_Omega, "p0": p0,
        "f_int": f_int, "share_corner": share_corner, "avg_rho": avg_rho,
        "max_drag": max_drag, "slope_rK_mid": slope_rK_mid,
        "d_rho_pos": d_rho_pos, "d_D_pos": d_D_pos, "d_rK_pos": d_rK_pos,
        "mono_violation_share_rho": mono_violation_share_rho,
        "mono_violation_share_D": mono_violation_share_D,
        "mono_violation_share_rK": mono_violation_share_rK,
        "interior_ok": interior_ok, "feasibility_ok": feasibility_ok, "soc_ok": soc_ok,
        "kkt_ok": kkt_ok, "micro_opt_ok": micro_opt_ok,
        "underpricing_regime": underpricing_regime,
        "domain_core_block1": domain_core_block1,
        "domain_outside_block1": domain_outside_block1,
        "drag_dominant": drag_dominant,
        "drag_share": drag_share,
        "spill_share": spill_share,
        "spill_share_low25": spill_share_low25,
        "spill_share_high25": spill_share_high25,
        "spill_share_low10": spill_share_low10,
        "spill_share_high10": spill_share_high10,
        "spill_at_mid": spill_at_mid,
        "psi_cutoff_spill": psi_cutoff_spill,
        "mixed_spill": mixed_spill,
        "spillover_dominant": spillover_dominant,
        "spillover_regime": spillover_regime if with_spillovers else "no_spillovers",
        "spill_params_active": with_spillovers,
        "zeta_min": zstats["zeta_min"],
        "zeta_med": zstats["zeta_med"],
        "zeta_max": zstats["zeta_max"],
        "zeta_share_lt1": zstats["zeta_share_lt1"],
        "underpriced_all_zeta": zstats["underpriced_all_zeta"],
        "zeta_masked_share": zstats.get("zeta_masked_share", np.nan),
        "zeta_valid_share": zstats.get("zeta_valid_share", np.nan),
        "zeta_drKnet_alignment": zeta_drKnet_alignment,
        "drKnet_min": drKnet_min,
        "drKnet_med": drKnet_med,
        "Wstruct_endpoints_straddle": wstruct_stats["endpoints_straddle"],
        "Wstruct_unique_crossing": wstruct_stats["unique_crossing"],
        "Wstruct_psi_bar_hat": wstruct_stats["psi_bar_hat"],
        "Wstruct_min": wstruct_stats["W_min"],
        "Wstruct_max": wstruct_stats["W_max"],
        "Wstruct_at_psiL": wstruct_stats["W_at_start"],
        "Wstruct_at_1": wstruct_stats["W_at_end"],
        "Wstruct_n_crossings": wstruct_stats["n_crossings"],
        "deriv_invalid_any": bool(deriv_invalid_any),
        "deriv_invalid_count": int(deriv_invalid_count),
        "deriv_invalid_idx_first": int(deriv_invalid_idx[0]) if deriv_invalid_idx else np.nan,
        # solver diagnostics
        "baseline_status": st0,
        "solver_status": solver_status.tolist(),
        "solver_ok_share": 1.0 - bad_share,
        "excluded_solver": excluded_solver,
        "included": included,
        "first_invalid_idx": first_invalid_idx,
        "count_nonfinite": count_nonfinite,
        "count_no_bracket": count_no_bracket,
        "count_no_converge": count_no_converge,
        "count_corner_low": count_corner_low,
        "count_corner_high": count_corner_high,
        "count_interior": count_interior,
    }

def run_block1_mc(rng, cfg, with_spillovers=False, with_premium=False, thetas=None):
    if thetas is None:
        rows = [run_block1_draw(rng, cfg, with_spillovers=with_spillovers, with_premium=with_premium) for _ in range(cfg["N"])]
    else:
        rows = [run_block1_draw(rng, cfg, theta=t, with_spillovers=with_spillovers, with_premium=with_premium) for t in thetas]
    return pd.DataFrame(rows)

def summarize_block1(df, label="core"):
    n_total = len(df)
    if n_total == 0:
        return pd.DataFrame([{
            "label": label, "N": 0,
            "share_d_rho_pos": np.nan,
            "share_d_D_pos": np.nan,
            "share_d_rK_pos": np.nan,
            "share_all_three": np.nan,
            "mean_f_int": np.nan,
            "mean_share_corner": np.nan,
            "mean_avg_rho": np.nan,
            "mean_max_drag": np.nan,
            "mean_slope_rK_mid": np.nan,
        }])
    summary = {
        "label": label,
        "N": n_total,
        "share_d_rho_pos": df["d_rho_pos"].mean(),
        "share_d_D_pos": df["d_D_pos"].mean(),
        "share_d_rK_pos": df["d_rK_pos"].mean(),
        "share_all_three": (df["d_rho_pos"] & df["d_D_pos"] & df["d_rK_pos"]).mean(),
        "mean_f_int": df["f_int"].mean(),
        "mean_share_corner": df["share_corner"].mean(),
        "mean_avg_rho": df["avg_rho"].mean(),
        "mean_max_drag": df["max_drag"].mean(),
        "mean_slope_rK_mid": df["slope_rK_mid"].mean(),
    }
    return pd.DataFrame([summary])

PARAMS_FOR_DIST = [
    "rho_bar", "a", "b", "r0",
    "c_lambda", "eta_lambda",
    "c_Delta", "eta_Delta",
    "c_delta", "eta_delta",
    "psi_L", "Wbar", "Gamma",
    "c_Omega", "eta_Omega", "p0",
]
DOMAIN_STATS_FOR_DIST = [
    "f_int", "share_corner", "avg_rho", "max_drag", "slope_rK_mid",
]

def _filter_existing(df, cols):
    return [c for c in cols if c in df.columns]

def _domain_summary_table(df_param, columns, domain_col="domain"):
    cols = _filter_existing(df_param, columns)
    if not cols:
        return pd.DataFrame()
    grouped = df_param.groupby(domain_col)[cols]
    summary = grouped.agg(["mean", "std", "min", "max", "median"])
    summary.columns = ["{}_{}".format(c, stat) for c, stat in summary.columns]
    summary = summary.reset_index()
    return summary

def _make_bins(core, outside, nbins=25):
    data = pd.concat([core, outside]).dropna()
    if data.empty:
        return None
    lo, hi = data.min(), data.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return None
    return np.linspace(lo, hi, nbins)

def plot_param_distributions(df_core, df_out, out_dir, mc_name, run_ts, params=None):
    if df_core.empty and df_out.empty:
        return {}
    params = _filter_existing(pd.concat([df_core, df_out], axis=0), params or PARAMS_FOR_DIST)
    if not params:
        return {}
    n = len(params)
    ncols = 3
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.2 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    legend_added = False
    for i, p in enumerate(params):
        ax = axes[i]
        core = df_core[p].dropna() if p in df_core.columns else pd.Series([], dtype=float)
        outside = df_out[p].dropna() if p in df_out.columns else pd.Series([], dtype=float)
        bins = _make_bins(core, outside, nbins=25)
        if bins is None:
            ax.text(0.5, 0.5, "degenerate", ha="center", va="center")
        else:
            if len(outside) > 0:
                ax.hist(outside, bins=bins, histtype="step", color="0.6", linestyle="--", linewidth=1.2, label="outside" if not legend_added else None)
            if len(core) > 0:
                ax.hist(core, bins=bins, histtype="step", color="0.2", linestyle="-", linewidth=1.5, label="core" if not legend_added else None)
                ax.axvline(core.median(), color="0.2", linewidth=1.0)
            if not legend_added and (len(core) > 0 or len(outside) > 0):
                legend_added = True
        ax.set_title(p)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    if legend_added:
        fig.legend(loc="upper right", frameon=False)
    fig.suptitle("MC Block 1 – Parameter distributions (core vs outside)")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    pdf_path = os.path.join(out_dir, f"{mc_name}_paramdist_{run_ts}.pdf")
    png_path = os.path.join(out_dir, f"{mc_name}_paramdist_{run_ts}.png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return {"plots_paramdist_pdf": pdf_path, "plots_paramdist_png": png_path}

def plot_domain_summaries(df_core, df_out, out_dir, mc_name, run_ts, stats=None):
    if df_core.empty and df_out.empty:
        return {}
    stats = _filter_existing(pd.concat([df_core, df_out], axis=0), stats or DOMAIN_STATS_FOR_DIST)
    if not stats:
        return {}
    n = len(stats)
    ncols = 2
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.6 * ncols, 3.2 * nrows))
    axes = np.atleast_1d(axes).ravel()
    legend_added = False
    for i, s in enumerate(stats):
        ax = axes[i]
        core = df_core[s].dropna() if s in df_core.columns else pd.Series([], dtype=float)
        outside = df_out[s].dropna() if s in df_out.columns else pd.Series([], dtype=float)
        bins = _make_bins(core, outside, nbins=25)
        if bins is None:
            ax.text(0.5, 0.5, "degenerate", ha="center", va="center")
        else:
            if len(outside) > 0:
                ax.hist(outside, bins=bins, histtype="step", color="0.6", linestyle="--", linewidth=1.2, label="outside" if not legend_added else None)
            if len(core) > 0:
                ax.hist(core, bins=bins, histtype="step", color="0.2", linestyle="-", linewidth=1.5, label="core" if not legend_added else None)
                ax.axvline(core.median(), color="0.2", linewidth=1.0)
            if not legend_added and (len(core) > 0 or len(outside) > 0):
                legend_added = True
        ax.set_title(s)
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")
    if legend_added:
        fig.legend(loc="upper right", frameon=False)
    fig.suptitle("MC Block 1 – Domain diagnostics (core vs outside)")
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    pdf_path = os.path.join(out_dir, f"{mc_name}_domainstats_{run_ts}.pdf")
    png_path = os.path.join(out_dir, f"{mc_name}_domainstats_{run_ts}.png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    plt.close(fig)
    return {"plots_domainstats_pdf": pdf_path, "plots_domainstats_png": png_path}

# ---------
# Run MCs
# ---------
rng_theta = np.random.default_rng(SEED)
shared_thetas = [draw_primitives_base(rng_theta, MC_CONFIG) for _ in range(MC_CONFIG["N"])]

rng_base = np.random.default_rng(SEED + 1)
rng_spill = np.random.default_rng(SEED + 2)
rng_prem = np.random.default_rng(SEED + 3)

df_block1 = run_block1_mc(rng_base, MC_CONFIG, with_spillovers=False, with_premium=False, thetas=shared_thetas)
df_spill = run_block1_mc(rng_spill, MC_CONFIG, with_spillovers=True, with_premium=False, thetas=shared_thetas)
df_prem = run_block1_mc(rng_prem, MC_CONFIG, with_spillovers=False, with_premium=True, thetas=shared_thetas)

def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    center = (p + z**2 / (2*n)) / denom
    half = (z * np.sqrt((p*(1-p) + z**2/(4*n)) / n)) / denom
    return (center - half, center + half)

def pct(x):
    return f"{100*x:6.2f}%" if pd.notna(x) else "  NA  "

OK_BASELINE = {"interior", "corner_low", "corner_high"}

def solver_audit_table(df, label):
    """
    Separates solver exclusions into:
      (A) baseline ψ=0 failure (baseline_status not OK_BASELINE)
      (B) grid failure (excluded_solver==True AND baseline ψ=0 OK)
    """
    n_total = len(df)
    if n_total == 0:
        return pd.DataFrame([{
            "run": label,
            "N_total": 0,
            "N_baseline_fail": 0,
            "N_grid_fail": 0,
            "N_excluded_solver": 0,
            "baseline_fail_share": np.nan,
            "grid_fail_share": np.nan,
            "excluded_solver_share": np.nan,
        }])

    bs = df["baseline_status"] if "baseline_status" in df.columns else pd.Series([np.nan]*n_total)
    baseline_ok = bs.isin(OK_BASELINE).fillna(False)
    baseline_fail = ~baseline_ok

    excluded_solver = df["excluded_solver"].fillna(False) if "excluded_solver" in df.columns else pd.Series([False]*n_total)
    grid_fail = excluded_solver & baseline_ok

    N_baseline_fail = int(baseline_fail.sum())
    N_grid_fail = int(grid_fail.sum())
    N_excluded = int(excluded_solver.sum())

    return pd.DataFrame([{
        "run": label,
        "N_total": n_total,
        "N_baseline_fail": N_baseline_fail,
        "N_grid_fail": N_grid_fail,
        "N_excluded_solver": N_excluded,
        "baseline_fail_share": N_baseline_fail / n_total,
        "grid_fail_share": N_grid_fail / n_total,
        "excluded_solver_share": N_excluded / n_total,
    }])

def baseline_status_breakdown(df, label):
    """
    Value counts of baseline_status (ψ=0 solve) for transparency.
    """
    if "baseline_status" not in df.columns:
        return pd.DataFrame([{"run": label, "baseline_status": "MISSING", "N": len(df)}])
    vc = df["baseline_status"].fillna("NA").value_counts(dropna=False).reset_index()
    vc.columns = ["baseline_status", "N"]
    vc.insert(0, "run", label)
    return vc

def grid_fail_reason_breakdown(df, label):
    ok = df["baseline_status"].isin(OK_BASELINE).fillna(False) if "baseline_status" in df.columns else pd.Series([False]*len(df))
    grid_fail = df["excluded_solver"].fillna(False) & ok if "excluded_solver" in df.columns else pd.Series([False]*len(df))
    if not grid_fail.any():
        return pd.DataFrame([{
            "run": label,
            "grid_fail_dominant_status_count": "NONE",
            "N": 0,
            "mean_count_nonfinite": np.nan,
            "mean_count_no_bracket": np.nan,
            "mean_count_no_converge": np.nan,
        }])

    reason_cols = [c for c in ["count_nonfinite", "count_no_bracket", "count_no_converge"] if c in df.columns]
    if not reason_cols:
        return pd.DataFrame([{
            "run": label,
            "grid_fail_dominant_status_count": "MISSING_COUNTS",
            "N": int(grid_fail.sum()),
            "mean_count_nonfinite": np.nan,
            "mean_count_no_bracket": np.nan,
            "mean_count_no_converge": np.nan,
        }])

    sub = df.loc[grid_fail, reason_cols].copy()
    sub_means = sub.mean().to_dict()
    reasons = sub.idxmax(axis=1).fillna("unknown")
    out = reasons.value_counts().reset_index()
    out.columns = ["grid_fail_dominant_status_count", "N"]
    out.insert(0, "run", label)
    out["mean_count_nonfinite"] = sub_means.get("count_nonfinite", np.nan)
    out["mean_count_no_bracket"] = sub_means.get("count_no_bracket", np.nan)
    out["mean_count_no_converge"] = sub_means.get("count_no_converge", np.nan)
    return out

def audit_consistency_check(audit_df):
    tmp = audit_df.copy()
    tmp["check_sum"] = tmp["N_baseline_fail"] + tmp["N_grid_fail"]
    tmp["check_ok"]  = (tmp["check_sum"] == tmp["N_excluded_solver"])
    if not tmp["check_ok"].all():
        bad = tmp.loc[~tmp["check_ok"], ["run","N_excluded_solver","N_baseline_fail","N_grid_fail","check_sum"]]
        print("\n!!! AUDIT CONSISTENCY FAIL (excluded != baseline + grid) !!!")
        print(bad.to_string(index=False))
    return tmp["check_ok"].all()

# ---------
# Output files
# ---------
def save_df(df, tag):
    fname = f"{MC_NAME}_{tag}_{RUN_TS}.csv"
    fpath = os.path.join(OUT_DIR, fname)
    df.to_csv(fpath, index=False)
    return fpath

paths = {}

# ----------------------------
# Solver audit: baseline ψ=0 vs grid failures
# ----------------------------
audit_rows = [
    solver_audit_table(df_block1, "block1"),
    solver_audit_table(df_spill,  "spill"),
    solver_audit_table(df_prem,   "prem"),
]
audit_solver = pd.concat(audit_rows, ignore_index=True)

audit_consistency_check(audit_solver)

print("\n=== Solver audit (baseline ψ=0 vs grid failures) ===")
print(audit_solver.to_string(index=False))

bs_breakdown = pd.concat([
    baseline_status_breakdown(df_block1, "block1"),
    baseline_status_breakdown(df_spill,  "spill"),
    baseline_status_breakdown(df_prem,   "prem"),
], ignore_index=True)

print("\n=== Baseline ψ=0 status breakdown ===")
print(bs_breakdown.to_string(index=False))

grid_reason = pd.concat([
    grid_fail_reason_breakdown(df_block1, "block1"),
    grid_fail_reason_breakdown(df_spill,  "spill"),
    grid_fail_reason_breakdown(df_prem,   "prem"),
], ignore_index=True)

print("\n=== Grid-fail dominant status count (conditional on baseline OK) ===")
print(grid_reason.to_string(index=False))

# Save
paths["audit_solver_b1"] = save_df(audit_solver, "audit_solver_b1")
paths["audit_baseline_status_breakdown_b1"] = save_df(bs_breakdown, "audit_baseline_status_breakdown_b1")
paths["audit_grid_fail_reasons_b1"] = save_df(grid_reason, "audit_grid_fail_reasons_b1")

df_block1_incl = df_block1[df_block1["included"]].copy()
df_spill_incl = df_spill[df_spill["included"]].copy()
df_prem_incl = df_prem[df_prem["included"]].copy()

df_core = df_block1_incl[df_block1_incl["domain_core_block1"]].copy()
df_out = df_block1_incl[~df_block1_incl["domain_core_block1"]].copy()

df_spill_core = df_spill_incl[df_spill_incl["domain_core_block1"]].copy()
df_spill_out = df_spill_incl[~df_spill_incl["domain_core_block1"]].copy()

df_prem_core = df_prem_incl[df_prem_incl["domain_core_block1"]].copy()
df_prem_out = df_prem_incl[~df_prem_incl["domain_core_block1"]].copy()

corner_threshold = 0.5
df_corner = df_block1_incl[df_block1_incl["share_corner"] >= corner_threshold].copy()

# ---------
# Accounting tables
# ---------
def accounting_table(df, core_col, label):
    n_total = len(df)
    n_excl = int(df["excluded_solver"].sum()) if "excluded_solver" in df.columns else 0
    n_incl = n_total - n_excl
    if n_incl > 0:
        df_incl = df[df["included"]]
        n_core = int(df_incl[core_col].sum())
        n_out = int((~df_incl[core_col]).sum())
    else:
        n_core = 0
        n_out = 0
    return {
        "run": label,
        "N_total": n_total,
        "N_excluded_solver": n_excl,
        "N_included": n_incl,
        "N_core_economic": n_core,
        "N_outside_economic": n_out,
    }

acct_rows = [
    accounting_table(df_block1, "domain_core_block1", "block1"),
    accounting_table(df_spill, "domain_core_block1", "spill"),
    accounting_table(df_prem, "domain_core_block1", "prem"),
 ]
acct_table = pd.DataFrame(acct_rows)
print("\n=== Accounting table (included vs excluded) ===")
print(acct_table.to_string(index=False))

# ---------
# Summaries
# ---------
core_summary = summarize_block1(df_core, label="core_baseline")
out_summary = summarize_block1(df_out, label="outside_baseline")
spill_summary = summarize_block1(df_spill_out[df_spill_out["spillover_dominant"]], label="spillover_dominant")
spill_core_summary = summarize_block1(df_spill_core, label="spillover_core_domain")

prem_over = df_prem_out[df_prem_out["p0"] >= 1.0]
prem_under = df_prem_core[df_prem_core["p0"] < 1.0]
prem_over_summary = summarize_block1(prem_over, label="over_internalized")
prem_under_summary = summarize_block1(prem_under, label="underpriced_core")
corner_summary = summarize_block1(df_corner, label="corner_heavy")

summary_table = pd.concat([
    core_summary, out_summary, spill_core_summary, spill_summary, prem_under_summary, prem_over_summary, corner_summary
], ignore_index=True)

# ---------
# Parameter / domain diagnostics by core vs outside
# ---------
df_param = df_block1_incl.copy()
df_param["domain"] = np.where(df_param["domain_core_block1"], "core", "outside")

param_summary = _domain_summary_table(df_param, PARAMS_FOR_DIST)
domain_stats_summary = _domain_summary_table(df_param, DOMAIN_STATS_FOR_DIST)

# ---------
# Output files
# ---------
paths.update({
    "block1_all": save_df(df_block1, "all"),
    "block1_core": save_df(df_core, "core"),
    "block1_out": save_df(df_out, "out"),
    "block1_spill": save_df(df_spill, "spill"),
    "block1_prem": save_df(df_prem, "prem"),
    "block1_corner": save_df(df_corner, "corner"),
    "summary": save_df(summary_table, "summary"),
    "param_summary_by_domain": save_df(param_summary, "param_summary_by_domain"),
    "domain_stats_summary": save_df(domain_stats_summary, "domain_stats_summary"),
    "accounting_table": save_df(acct_table, "accounting_table"),
})

# ---------
# Grayscale plots
# ---------
paths.update(plot_param_distributions(df_core, df_out, OUT_DIR, MC_NAME, RUN_TS))
paths.update(plot_domain_summaries(df_core, df_out, OUT_DIR, MC_NAME, RUN_TS))

# ---------
# Console output
# ---------
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

print("\n" + "=" * 100)
print(f"MC Block 1: {MC_NAME} | timestamp={RUN_TS} | N={MC_CONFIG['N']} | psi_grid={MC_CONFIG['psi_grid_n']}")
print("Domain rules: interior_ok (f_int>=tau), feasibility_ok, soc_ok, kkt_ok")
print(f"tau_int={MC_CONFIG['tau_int']}, corner_threshold={corner_threshold}")
print("=" * 100)

def rate(x):
    return f"{100.0*x:6.2f}%" if pd.notna(x) else "  NA  "

def show_block(label, df):
    n = len(df)
    if n == 0:
        print(f"\n[{label}] N=0")
        return
    share_pos = (df["d_rho_pos"] & df["d_D_pos"] & df["d_rK_pos"]).mean()
    print(f"\n[{label}] N={n}")
    print(f"  share(dρ/dψ>=-tol): {rate(df['d_rho_pos'].mean())}")
    print(f"  share(𝒟' >=-tol)  : {rate(df['d_D_pos'].mean())}")
    print(f"  share(rK' >=-tol) : {rate(df['d_rK_pos'].mean())}")
    print(f"  share(all three)  : {rate(share_pos)}")
    print(f"  mean f_int        : {df['f_int'].mean():.3f}")
    print(f"  mean share_corner : {df['share_corner'].mean():.3f}")
    print(f"  mean avg ρ*       : {df['avg_rho'].mean():.3f}")
    print(f"  mean max 𝒟        : {df['max_drag'].mean():.3f}")
    print(f"  mean slope rK(mid): {df['slope_rK_mid'].mean():.3f}")

show_block("CORE BASELINE", df_core)
show_block("OUTSIDE BASELINE", df_out)
show_block("SPILLOVER CORE", df_spill_core)
show_block("SPILLOVER DOMINANT", df_spill_out[df_spill_out["spillover_dominant"]])
show_block("UNDERPRICED CORE", prem_under)
show_block("OVER-INTERNALIZED", prem_over)
show_block("CORNER-HEAVY", df_corner)

print("\nSaved outputs:")
for k, v in paths.items():
    print(f"  {k:24s} -> {v}")

print("\nSummary table preview:")
with pd.option_context("display.max_rows", 20, "display.max_columns", 20):
    print(summary_table)

print("\nParameter summary by domain (preview):")
with pd.option_context("display.max_rows", 10, "display.max_columns", 20):
    print(param_summary.head(10))

print("\nDomain stats summary (preview):")
with pd.option_context("display.max_rows", 10, "display.max_columns", 20):
    print(domain_stats_summary.head(10))

# ----------------------------
# Post-run diagnostics
# ----------------------------
df_all = df_block1_incl.copy()
df_all_flags = df_all.copy()
df_all_flags["outside"] = ~df_all_flags["domain_core_block1"]

# 1) Core share with CI (included denominator)
n_total = len(df_block1)
n_excl = int(df_block1["excluded_solver"].sum())
n_incl = n_total - n_excl
n_core = len(df_core)
core_share = n_core / n_incl if n_incl else np.nan
ci_lo, ci_hi = wilson_ci(n_core, n_incl)
excluded_share = n_excl / n_total if n_total else np.nan

print("\n=== Core share (Wilson 95% CI, included only) ===")
print(f"Core share: {pct(core_share)} (N_core={n_core}/N_included={n_incl}, CI: {pct(ci_lo)}–{pct(ci_hi)})")
print(f"Excluded (solver invalid): {pct(excluded_share)} (N_excluded={n_excl}/N_total={n_total})")

# 2) Failure-mode breakdown (outside domain, included only)
flags = ["interior_ok", "feasibility_ok", "soc_ok", "kkt_ok", "underpricing_regime", "drag_dominant"]
fail_rows = []
for f in flags:
    if f in df_all_flags.columns:
        fail_all = (~df_all_flags[f]).mean()
        fail_out = (~df_all_flags.loc[df_all_flags["outside"], f]).mean() if df_all_flags["outside"].any() else np.nan
        fail_rows.append({"flag": f, "fail_share_all": fail_all, "fail_share_outside": fail_out})

fail_table = pd.DataFrame(fail_rows)
print("\n=== Failure-mode shares (included only) ===")
print(fail_table.to_string(index=False))

# 3) Parameter associations with outside indicator (Spearman)
param_cols = [
    "rho_bar","a","b","c_lambda","eta_lambda","c_Delta","eta_Delta","c_delta","eta_delta","psi_L","Wbar","Gamma","c_Omega","eta_Omega","p0"
 ]
param_cols = [c for c in param_cols if c in df_all.columns]
assoc = []
if param_cols:
    outside_int = df_all_flags["outside"].astype(int)
    for c in param_cols:
        corr = df_all_flags[[c]].join(outside_int.rename("outside")).corr(method="spearman").iloc[0,1]
        assoc.append({"param": c, "spearman_with_outside": corr})
assoc_df = pd.DataFrame(assoc).sort_values(by="spearman_with_outside", key=lambda s: s.abs(), ascending=False)
print("\n=== Spearman correlation with outside indicator ===")
print(assoc_df.to_string(index=False))

# 4) Spillover-dominant prevalence (in spill run, included only)
if "spillover_dominant" in df_spill_incl.columns:
    spill_share = df_spill_incl["spillover_dominant"].mean()
    n_spill = len(df_spill_incl)
    c_lo, c_hi = wilson_ci(int(df_spill_incl["spillover_dominant"].sum()), n_spill)
    print("\n=== Spillover-dominant prevalence (spill run, included only) ===")
    print(f"Share: {pct(spill_share)} (N={n_spill}, CI: {pct(c_lo)}–{pct(c_hi)})")
else:
    spill_share = np.nan
    n_spill = len(df_spill_incl)
    c_lo, c_hi = (np.nan, np.nan)

# 5) Quick comparison of key parameters by domain
key_params = [c for c in ["rho_bar","psi_L","Wbar","c_Omega","eta_Omega","p0"] if c in df_all.columns]
if key_params:
    comp = df_all.assign(domain=np.where(df_all["domain_core_block1"], "core", "outside")).groupby("domain")[key_params].median().reset_index()
    print("\n=== Median parameters by domain ===")
    print(comp.to_string(index=False))
else:
    comp = pd.DataFrame()

# ----------------------------
# Out-of-domain primitive vector summary
# ----------------------------
primitive_cols = [
    "rho_bar","a","b","c_lambda","eta_lambda","c_Delta","eta_Delta","c_delta","eta_delta","psi_L","Wbar","Gamma","c_Omega","eta_Omega","p0"
 ]
primitive_cols = [c for c in primitive_cols if c in df_all.columns]

print("\n=== Out-of-domain primitive vector summary ===")
if not primitive_cols:
    print("No primitive columns found.")
    q = pd.DataFrame()
else:
    q = df_out[primitive_cols].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]).T
    q.columns = ["p01","p05","p25","p50","p75","p95","p99"]
    q["mean"] = df_out[primitive_cols].mean()
    q["std"] = df_out[primitive_cols].std()
    q = q[["mean","std","p01","p05","p25","p50","p75","p95","p99"]]
    print(q.to_string())

# Compare out-of-domain to core: standardized distance using core mean/std
print("\n=== Out-of-domain vs core: standardized deviation ===")
if primitive_cols and not df_core.empty:
    core_mean = df_core[primitive_cols].mean()
    core_std = df_core[primitive_cols].std().replace(0.0, np.nan)
    out_mean = df_out[primitive_cols].mean()
    z = (out_mean - core_mean) / core_std
    z = z.sort_values(key=lambda s: s.abs(), ascending=False)
    z_table = pd.DataFrame({"z_out_minus_core": z})
    print(z_table.to_string())
else:
    z_table = pd.DataFrame()
    print("Core sample empty or no primitive columns.")

# Flag out-of-domain extremes relative to core distribution (|z| >= 2)
print("\n=== Out-of-domain extreme rates vs core (|z|>=2) ===")
if primitive_cols and not df_core.empty:
    core_mean = df_core[primitive_cols].mean()
    core_std = df_core[primitive_cols].std().replace(0.0, np.nan)
    z_out = (df_out[primitive_cols] - core_mean) / core_std
    extreme_rate = (z_out.abs() >= 2.0).mean().sort_values(ascending=False)
    extreme_table = pd.DataFrame({"share_|z|>=2": extreme_rate})
    print(extreme_table.to_string())
else:
    extreme_table = pd.DataFrame()
    print("Core sample empty or no primitive columns.")

# ----------------------------
# Save diagnostic outputs
# ----------------------------
def save_diag(df, tag):
    fname = f"{MC_NAME}_{tag}_{RUN_TS}.csv"
    fpath = os.path.join(OUT_DIR, fname)
    if df is None:
        df = pd.DataFrame()
    df.to_csv(fpath, index=False)
    return fpath

def save_excluded(df, tag, primitive_cols):
    cols = primitive_cols + [
        "first_invalid_idx","count_nonfinite","count_no_bracket","count_no_converge",
        "count_corner_low","count_corner_high","count_interior"
    ]
    cols = [c for c in cols if c in df.columns]
    return save_diag(df[cols].copy(), tag)

diag_paths = {}
diag_paths["diag_failure_shares"] = save_diag(fail_table, "diag_failure_shares")
diag_paths["diag_spearman_outside"] = save_diag(assoc_df, "diag_spearman_outside")
diag_paths["diag_median_by_domain"] = save_diag(comp, "diag_median_by_domain")
diag_paths["diag_outdomain_quantiles"] = save_diag(q, "diag_outdomain_quantiles")
diag_paths["diag_outdomain_z"] = save_diag(z_table, "diag_outdomain_z")
diag_paths["diag_outdomain_extreme_rates"] = save_diag(extreme_table, "diag_outdomain_extreme_rates")

diag_core_share = pd.DataFrame([{
    "core_share": core_share,
    "core_share_ci_lo": ci_lo,
    "core_share_ci_hi": ci_hi,
    "excluded_share": excluded_share,
    "N_total": n_total,
    "N_included": n_incl,
    "N_excluded": n_excl,
    "N_core": n_core,
}])
diag_paths["diag_core_share"] = save_diag(diag_core_share, "diag_core_share")
diag_paths["diag_excluded_block1"] = save_excluded(df_block1[df_block1["excluded_solver"]], "diag_excluded_block1", primitive_cols)
diag_paths["diag_excluded_spill"] = save_excluded(df_spill[df_spill["excluded_solver"]], "diag_excluded_spill", primitive_cols)
diag_paths["diag_excluded_prem"] = save_excluded(df_prem[df_prem["excluded_solver"]], "diag_excluded_prem", primitive_cols)

print("\nSaved diagnostic outputs:")
for k, v in diag_paths.items():
    print(f"  {k:28s} -> {v}")

# =========================
# Block 1c – MP corridor MC run and stats
# =========================

print("\n" + "=" * 100)
print(f"MC Block 1c (MP corridor): {MC_NAME} | timestamp={RUN_TS} | N={MC_CONFIG['N']}")
print("=" * 100)

rng_mp = np.random.default_rng(SEED + 4)
df_1c = run_block1c_mc(rng_mp, MC_CONFIG, thetas=shared_thetas)
df_1c = df_1c.reset_index(drop=True)
df_1c["idx"] = df_1c.index
df_1c_incl = df_1c[df_1c["included"]].copy()

mp_paths = {}

audit_solver_1c = solver_audit_table(df_1c, "block1c")
audit_solver = pd.concat([audit_solver, audit_solver_1c], ignore_index=True)

audit_consistency_check(audit_solver)

bs_breakdown_1c = baseline_status_breakdown(df_1c, "block1c")
bs_breakdown = pd.concat([bs_breakdown, bs_breakdown_1c], ignore_index=True)

grid_reason_1c = grid_fail_reason_breakdown(df_1c, "block1c")
grid_reason = pd.concat([grid_reason, grid_reason_1c], ignore_index=True)

print("\n=== Solver audit (including block1c) ===")
print(audit_solver.to_string(index=False))

print("\n=== Baseline ψ=0 status breakdown (including block1c) ===")
print(bs_breakdown.to_string(index=False))

print("\n=== Grid-fail dominant status count (including block1c) ===")
print(grid_reason.to_string(index=False))

mp_paths["audit_solver_all"] = save_df(audit_solver, "audit_solver_all")
mp_paths["audit_baseline_status_breakdown_all"] = save_df(bs_breakdown, "audit_baseline_status_breakdown_all")
mp_paths["audit_grid_fail_reasons_all"] = save_df(grid_reason, "audit_grid_fail_reasons_all")

# MP domain labels
df_1c_incl["mp_domain"] = np.where(df_1c_incl["mp_domain_core"], "mp_core", "not_mp_core")

N_total_1c   = len(df_1c)
N_excl_1c    = int(df_1c["excluded_solver"].sum())
N_incl_1c    = N_total_1c - N_excl_1c
N_mp_core    = int(df_1c_incl["mp_domain_core"].sum())
share_mp_core = N_mp_core / N_incl_1c if N_incl_1c > 0 else 0.0

print(f"\nMP-core share (Block 1c, included only): {share_mp_core*100:6.2f}% (N={N_mp_core}/{N_incl_1c})")
print(f"Excluded (solver invalid): {100*N_excl_1c/N_total_1c:6.2f}% (N={N_excl_1c}/{N_total_1c})")

# Parameter summary for MP vs non-MP
MP_PARAMS_FOR_DIST = [
    "rho_bar","a","b","c_lambda","eta_lambda",
    "c_Delta","eta_Delta","c_delta","eta_delta",
    "psi_L","Wbar","Gamma","kappa_c"
 ]
param_summary_mp = _domain_summary_table(df_1c_incl, MP_PARAMS_FOR_DIST, domain_col="mp_domain")

print("\nParameter summary by MP domain (mp_core vs not_mp_core):")
print(param_summary_mp)

mp_diag = df_1c_incl.groupby("mp_domain")[["mp_corr_share", "corridor_corner_share"]].agg(["mean", "median"]).reset_index()
save_df(mp_diag, "mp_corr_and_corner_share_by_domain")

# Detailed Gamma / Wbar / kappa_c quantiles
def param_quantile_summary(df, param, label_col="mp_domain"):
    rows = []
    for lab in ["all", "mp_core", "not_mp_core"]:
        if lab == "all":
            s = df[param].dropna()
        elif lab == "mp_core":
            s = df.loc[df["mp_domain_core"], param].dropna()
        else:
            s = df.loc[~df["mp_domain_core"], param].dropna()
        if s.empty:
            rows.append({
                "group": lab, "N": 0,
                "mean": np.nan, "std": np.nan,
                "p01": np.nan, "p05": np.nan, "p25": np.nan,
                "p50": np.nan, "p75": np.nan, "p95": np.nan, "p99": np.nan
            })
        else:
            pct = np.percentile(s, [1,5,25,50,75,95,99])
            rows.append({
                "group": lab, "N": len(s),
                "mean": s.mean(), "std": s.std(),
                "p01": pct[0], "p05": pct[1], "p25": pct[2],
                "p50": pct[3], "p75": pct[4], "p95": pct[5], "p99": pct[6],
            })
    return pd.DataFrame(rows)

gamma_summary = param_quantile_summary(df_1c_incl, "Gamma")
Wbar_summary  = param_quantile_summary(df_1c_incl, "Wbar")
kappa_summary = param_quantile_summary(df_1c_incl, "kappa_c")

print("\nGamma distribution by MP domain:")
print(gamma_summary.to_string(index=False))

print("\nWbar distribution by MP domain:")
print(Wbar_summary.to_string(index=False))

print("\nkappa_c distribution by MP domain:")
print(kappa_summary.to_string(index=False))

# Save Block 1c outputs
mp_paths.update({
    "block1c_all": save_df(df_1c, "mp_all"),
    "block1c_core": save_df(df_1c_incl[df_1c_incl["mp_domain_core"]], "mp_core"),
    "block1c_notcore": save_df(df_1c_incl[~df_1c_incl["mp_domain_core"]], "mp_notcore"),
    "block1c_param_summary_by_mp": save_df(param_summary_mp, "mp_param_summary_by_mp"),
    "block1c_gamma_summary": save_df(gamma_summary, "mp_gamma_summary"),
    "block1c_Wbar_summary": save_df(Wbar_summary, "mp_Wbar_summary"),
    "block1c_kappa_summary": save_df(kappa_summary, "mp_kappa_summary"),
    "block1c_excluded": save_excluded(df_1c[df_1c["excluded_solver"]], "diag_excluded_block1c", MP_PARAMS_FOR_DIST),
})

print("\nSaved Block 1c (MP corridor) outputs:")
for k, v in mp_paths.items():
    print(f"  {k:32s} -> {v}")

# Append Block 1c accounting
acct_rows.append({
    "run": "block1c",
    "N_total": N_total_1c,
    "N_excluded_solver": N_excl_1c,
    "N_included": N_incl_1c,
    "N_core_economic": N_mp_core,
    "N_outside_economic": int((~df_1c_incl["mp_domain_core"]).sum())
})
acct_table = pd.DataFrame(acct_rows)
save_df(acct_table, "accounting_table")

# =========================
# Coupled diagnostics
# =========================

# Merge Block 1 baseline with Block 1c (MP corridor) on shared primitives index
df_b1 = df_block1.reset_index(drop=True).copy()
df_mp = df_1c.reset_index(drop=True).copy()
df_b1["idx"] = df_b1.index
df_mp["idx"] = df_mp.index

df_merge = df_b1.merge(
    df_mp[["idx", "included", "excluded_solver", "mp_domain_core", "mp_exists", "mp_corr_share", "wedge_regime"]],
    on="idx", how="left", suffixes=("_b1", "_mp")
 )

df_merge["both_included"] = df_merge["included_b1"] & df_merge["included_mp"]
df_merge_incl = df_merge[df_merge["both_included"]].copy()

# Overlap table (included-only)
overlap = pd.crosstab(
    df_merge_incl["domain_core_block1"],
    df_merge_incl["mp_domain_core"],
    normalize="all"
 )

print("\n=== Overlap: Block 1 core vs MP core (included only shares) ===")
print(overlap)

# Conditional rates
def rate_cond(numer, denom):
    return (numer / denom) if denom else np.nan

n_total = len(df_merge_incl)
n_b1_core = int(df_merge_incl["domain_core_block1"].sum())
n_mp_core = int(df_merge_incl["mp_domain_core"].sum())
n_both_core = int((df_merge_incl["domain_core_block1"] & df_merge_incl["mp_domain_core"]).sum())
n_b1_only = int((df_merge_incl["domain_core_block1"] & ~df_merge_incl["mp_domain_core"]).sum())
n_mp_only = int((~df_merge_incl["domain_core_block1"] & df_merge_incl["mp_domain_core"]).sum())

print("\n=== Conditional overlap rates (included only) ===")
print(f"P(MP core | Block1 core)      : {rate_cond(n_both_core, n_b1_core):.4f}")
print(f"P(Block1 core | MP core)      : {rate_cond(n_both_core, n_mp_core):.4f}")
print(f"Block1 core only share        : {rate_cond(n_b1_only, n_total):.4f}")
print(f"MP core only share            : {rate_cond(n_mp_only, n_total):.4f}")

# Compare MP corridor stats across Block1 core/outside
mp_corr_summary = df_merge_incl.groupby("domain_core_block1")["mp_corr_share"].agg(["mean", "median", "min", "max"]).reset_index()
print("\n=== MP corridor share by Block1 core (included only) ===")
print(mp_corr_summary)

# Wedge regime breakdown
if "wedge_regime" in df_merge_incl.columns:
    print("\n=== Wedge regime breakdown (included only) ===")
    print(df_merge_incl["wedge_regime"].value_counts(normalize=True))

# Spearman correlation between Block1 domain indicator and MP corridor length/share
corr_mp = df_merge_incl[["domain_core_block1", "mp_corr_share"]].corr(method="spearman").iloc[0,1]
print("\n=== Spearman corr: Block1 core vs MP corridor share (included only) ===")
print(corr_mp)

# Save coupled diagnostics
def save_diag(df, tag):
    fname = f"{MC_NAME}_{tag}_{RUN_TS}.csv"
    fpath = os.path.join(OUT_DIR, fname)
    if df is None:
        df = pd.DataFrame()
    df.to_csv(fpath, index=False)
    return fpath

overlap_table = overlap.reset_index()
diag_paths = {}
diag_paths["diag_overlap_shares_included"] = save_diag(overlap_table, "diag_overlap_shares_included")
diag_paths["diag_overlap_group_summary_included"] = save_diag(mp_corr_summary, "diag_overlap_group_summary_included")
diag_paths["diag_overlap_flags_included"] = save_diag(
    df_merge_incl[["idx", "included_b1", "included_mp", "domain_core_block1", "mp_domain_core", "mp_corr_share", "mp_exists", "wedge_regime"]],
    "diag_overlap_flags_included"
 )

print("\nSaved coupled diagnostics:")
for k, v in diag_paths.items():
    print(f"  {k:28s} -> {v}")